# 02 — Set Up Delta Sharing

Creates a **share** and **recipient** in Unity Catalog, adds the combine_results table
to the share, and downloads the recipient's activation profile JSON so the consumer
notebook can read it without any Databricks credentials.

In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "nfl_combine")
TABLE_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.combine_results"

SHARE_NAME     = "nfl_combine_share"
RECIPIENT_NAME = "nfl_combine_external_reader"

## Step 1 — Create the Share and add the table

In [2]:
# Create share (idempotent)
spark.sql(f"CREATE SHARE IF NOT EXISTS {SHARE_NAME}")

# Add the table to the share
# DROP first to make this idempotent, then re-add
try:
    spark.sql(f"ALTER SHARE {SHARE_NAME} REMOVE TABLE {TABLE_NAME}")
except Exception:
    pass

spark.sql(f"ALTER SHARE {SHARE_NAME} ADD TABLE {TABLE_NAME}")

# Verify
spark.sql(f"SHOW ALL IN SHARE {SHARE_NAME}").show(truncate=False)
print(f"Share '{SHARE_NAME}' is ready with table {TABLE_NAME}")

+---------------------------+-----+-------------------------------------------+-----------------------+------------------------------+-------+----------+---------------+----------+-------------+
|name                       |type |shared_object                              |added_at               |added_by                      |comment|partitions|history_sharing|cdf_shared|start_version|
+---------------------------+-----+-------------------------------------------+-----------------------+------------------------------+-------+----------+---------------+----------+-------------+
|nfl_combine.combine_results|TABLE|alexander_booth.nfl_combine.combine_results|2026-03-16 21:26:49.174|alexander.booth@databricks.com|NULL   |NULL      |ENABLED        |false     |0            |
+---------------------------+-----+-------------------------------------------+-----------------------+------------------------------+-------+----------+---------------+----------+-------------+

Share 'nfl_combine_share

## Step 2 — Create the Recipient and grant access

In [3]:
# Create recipient (open sharing — token-based, no Databricks account needed)
try:
    spark.sql(f"DROP RECIPIENT IF EXISTS {RECIPIENT_NAME}")
except Exception:
    pass

result = spark.sql(f"CREATE RECIPIENT {RECIPIENT_NAME}").collect()

# Grant SELECT on the share to the recipient
spark.sql(f"GRANT SELECT ON SHARE {SHARE_NAME} TO RECIPIENT {RECIPIENT_NAME}")

print(f"Recipient '{RECIPIENT_NAME}' created and granted access to '{SHARE_NAME}'")

Recipient 'nfl_combine_external_reader' created and granted access to 'nfl_combine_share'


## Step 3 — Download the activation profile

The profile JSON is what the **consumer** uses — it contains the sharing server
endpoint and a bearer token. No Databricks credentials needed on the consumer side.

In [4]:
import requests

# Get the activation URL from the recipient info
recipient_info = w.recipients.get(RECIPIENT_NAME)

activation_url = str(recipient_info.tokens[0].activation_url) if recipient_info.tokens else None

if activation_url and activation_url != "None":
    # The activation URL points to an HTML page. The actual profile JSON
    # is available via the REST activation API endpoint.
    base_url = activation_url.split("/delta_sharing/retrieve_config.html?")[0]
    token_part = activation_url.split("?")[1]
    api_url = f"{base_url}/api/2.0/unity-catalog/public/data_sharing_activation/{token_part}"

    resp = requests.get(api_url)
    resp.raise_for_status()
    profile = resp.json()

    profile_path = "share_profile.json"
    with open(profile_path, "w") as f:
        json.dump(profile, f, indent=2)

    print(f"Profile saved to {profile_path}")
    print(f"Sharing server: {profile.get('endpoint', 'N/A')}")
    print(f"Expires: {profile.get('expirationTime', 'N/A')}")
else:
    # If the recipient was already activated, we can't re-download.
    print("Recipient already activated — retrieve profile from the Databricks UI")
    print("Or re-run this notebook after dropping and recreating the recipient.")

Profile saved to share_profile.json
Sharing server: https://oregon.cloud.databricks.com/api/2.0/delta-sharing/metastores/b169b504-4c54-49f2-bc3a-adf4b128f36d
Expires: 2026-06-24T21:27:11.261Z


In [5]:
# Verify the share is visible to the recipient
spark.sql(f"SHOW GRANTS ON SHARE {SHARE_NAME}").show(truncate=False)

+---------------------------+---------+
|recipient                  |privilege|
+---------------------------+---------+
|nfl_combine_external_reader|SELECT   |
+---------------------------+---------+

